# Chess Dataset — Data Acquisition & Parsing
### CMPS344 Applied Data Science — Phase 2
---
**Sources:**
- `data.pgn` — 50,000 chess games in standard PGN notation
- `data_uci.pgn` — Same games with moves in UCI coordinate notation
- `stockfish.csv` — Per-game Stockfish centipawn evaluations

**Goal:** Parse all three files, extract features, merge into a single clean DataFrame, and save as `games.csv`.

## 1. Imports & Configuration

In [1]:
# uncomment this line to install dependencies (first run only)
# %pip install python-chess requests
import sys
import os

# Adds the parent directory to the python path
sys.path.append(os.path.abspath(os.path.join('..')))

In [2]:
import re
import io
import requests
import pandas as pd
import numpy as np
import chess
import chess.pgn
from collections import Counter
from src.data.load_data import *
from src.features.build_features import *

# ── Paths (update to match your folder structure) ────────────────────────────
PGN_PATH              = "../data/raw/data.pgn"
UCI_PATH              = "../data/raw/data_uci.pgn"
SF_PATH               = "../data/raw/stockfish.csv"
LICHESS_PATH          = "../data/raw/chess_games.csv"
LICHESS_SF_PATH       = "../data/raw/lichess_stockfish.csv"
OUTPUT_PATH           = "../data/processed/games.csv"
OUTPUT_MERGED_PATH    = "../data/processed/merged_games.csv"
ECO_PATH              = "../data/interim/eco_openings.csv"

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Download ECO Opening Database (Third Data Source)
Downloads the official Lichess opening database (public domain) from GitHub.
Files `a.tsv` through `e.tsv` cover all ECO codes A00–E99.

This is our **third data source** — merged with the PGN and Stockfish data to add
opening names (e.g. "Sicilian Defense: Najdorf Variation") and ECO codes per game.

> **Citation:** lichess-org/chess-openings, https://github.com/lichess-org/chess-openings (public domain)

In [3]:
# ── Load from disk if already downloaded, otherwise fetch ────────────────────
import os
if os.path.exists(ECO_PATH):
    print(f"Found cached '{ECO_PATH}', loading from disk...")
    df_eco = pd.read_csv(ECO_PATH)
    df_eco["moves_normalised"] = df_eco["moves_normalised"].fillna("")
    print(f"Loaded {len(df_eco):,} ECO entries.")
else:
    df_eco = download_eco_database(ECO_PATH)

df_eco.head(5)

Found cached '../data/interim/eco_openings.csv', loading from disk...
Loaded 3,641 ECO entries.


,eco,name,pgn,eco_family,moves_normalised
0,A00,Amar Opening,1. Nh3,A,Nh3
1,A00,Amar Opening: Paris Gambit,1. Nh3 d5 2. g3 e5 3. f4,A,Nh3 d5 g3 e5 f4
2,A00,"Amar Opening: Paris Gambit, Gent Gambit",1. Nh3 d5 2. g3 e5 3. f4 Bxh3 4. Bxh3 exf4 5. ...,A,Nh3 d5 g3 e5 f4 Bxh3 Bxh3 exf4 O-O fxg3 hxg3
3,A00,Amsterdam Attack,1. e3 e5 2. c4 d6 3. Nc3 Nc6 4. b3 Nf6,A,e3 e5 c4 d6 Nc3 Nc6 b3 Nf6
4,A00,Anderssen's Opening,1. a3,A,a3


## 3. Parse `data.pgn` with `python-chess`
Uses the `python-chess` library instead of regex for robust, standards-compliant parsing.

**Extra features now extracted vs the old regex parser:**

| Feature | Description |
|---|---|
| `white_castled` | Did White castle? (bool) |
| `black_castled` | Did Black castle? (bool) |
| `white_castle_side` | 'kingside', 'queenside', or 'none' |
| `black_castle_side` | 'kingside', 'queenside', or 'none' |
| `termination` | 'checkmate', 'resignation', 'draw', or 'unknown' |
| `num_captures` | Total captures in the game |

In [4]:
df_pgn = parse_pgn(PGN_PATH)
df_pgn.head(3)

Parsed 25,000 games from ../data/raw/data.pgn

Result distribution:
result
1-0        9704
1/2-1/2    7791
0-1        7505

Termination distribution:
termination
resignation    16653
draw            7790
checkmate        557


,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,termination,moves_san
0,1,2354,2411,1/2-1/2,19,True,True,kingside,kingside,9,draw,Nf3 Nf6 c4 c5 b3 g6 Bb2 Bg7 e3 O-O Be2 b6 O-O ...
1,2,2523,2460,1/2-1/2,6,True,False,kingside,none,2,draw,e4 e5 Nf3 Nf6 d4 Nxe4 Nxe5 d6 Nf3 d5 Bd3 Nd6 O-O
2,3,1915,1999,0-1,53,True,True,kingside,kingside,23,resignation,e4 d5 exd5 Nf6 d4 Nxd5 Nf3 g6 Be2 Bg7 c4 Nb6 N...


## 4. Match Games to ECO Openings
For each game, try to match the opening moves against the ECO database using
a **longest-prefix match** — the more moves played in the opening, the more
specific the ECO code we can assign.

In [5]:
# ── Build lookup and apply to all games ──────────────────────────────────────
print("Building ECO lookup table...")
eco_lookup = build_eco_lookup(df_eco)
print(f"Lookup table size: {len(eco_lookup):,} entries")

print("\nMatching games to ECO codes (this may take a minute)...")
eco_results = df_pgn["moves_san"].apply(lambda m: match_eco(m, eco_lookup))

df_pgn["eco_code"]     = eco_results.apply(lambda x: x[0])
df_pgn["opening_name"] = eco_results.apply(lambda x: x[1])
df_pgn["eco_family"]   = eco_results.apply(lambda x: x[2])

matched = (df_pgn["eco_code"] != "Unknown").sum()
print(f"\nGames matched to ECO opening: {matched:,} / {len(df_pgn):,} ({matched/len(df_pgn)*100:.1f}%)")
print(f"\nTop 10 openings:")
print(df_pgn["opening_name"].value_counts().head(10).to_string())

Building ECO lookup table...
Lookup table size: 3,641 entries

Matching games to ECO codes (this may take a minute)...

Games matched to ECO opening: 25,000 / 25,000 (100.0%)

Top 10 openings:
opening_name
Zukertort Opening                         1063
Sicilian Defense: Najdorf Variation        680
Pirc Defense                               394
Horwitz Defense                            368
Indian Defense: Knights Variation          357
Zukertort Opening: Sicilian Invitation     294
Sicilian Defense: Modern Variations        289
Ruy Lopez: Closed                          283
Caro-Kann Defense                          282
Indian Defense: Anti-Nimzo-Indian          279


## 5. Parse `data_uci.pgn`
Extracts move sequences in UCI coordinate format (e.g. `e2e4 e7e5`).
We keep this as a separate feature — UCI notation is useful for pattern matching and
is a distinct representation from the SAN moves already parsed from `data.pgn`.

In [6]:
df_uci = parse_uci(UCI_PATH)
df_uci.head(3)

Parsed 50,000 games from ../data/raw/data_uci.pgn


,event_id,moves_uci
0,1,g1f3 g8f6 c2c4 c7c5 b2b3 g7g6 c1b2 f8g7 e2e3 e...
1,2,e2e4 e7e5 g1f3 g8f6 d2d4 f6e4 f3e5 d7d6 e5f3 d...
2,3,e2e4 d7d5 e4d5 g8f6 d2d4 f6d5 g1f3 g7g6 f1e2 f...


## 6. Parse `stockfish.csv` & Extract Evaluation Features
Each row contains space-separated centipawn scores for every half-move in the game.
- **Positive score** = advantage for White
- **Negative score** = advantage for Black

We compute the following features per game:

| Feature | Description |
|---|---|
| `white_acl` | Average centipawn loss per move for White |
| `black_acl` | Average centipawn loss per move for Black |
| `white_blunders` | Moves where White lost ≥ 100 centipawns |
| `black_blunders` | Moves where Black lost ≥ 100 centipawns |
| `white_mistakes` | Moves where White lost 50–99 centipawns |
| `black_mistakes` | Moves where Black lost 50–99 centipawns |
| `final_eval` | Centipawn evaluation at game's last move |
| `max_white_advantage` | Peak advantage White held during the game |
| `max_black_advantage` | Peak advantage Black held (most negative value) |
| `game_sharpness` | Std deviation of all scores — higher = more tactical |

In [7]:
df_sf = extract_stockfish_features(SF_PATH)
df_sf.head(3)

Extracted Stockfish features for 47,174 games
       white_acl  black_acl  white_blunders  black_blunders
count   47174.00   47174.00        47174.00        47174.00
mean       57.14      63.49            1.44            1.61
std       100.39     107.61            1.97            2.06
min         0.00       0.00            0.00            0.00
25%        16.90      17.52            0.00            0.00
50%        25.73      28.62            1.00            1.00
75%        48.11      56.15            2.00            3.00
max      2111.92    1880.20           23.00           21.00


,event_id,total_half_moves,white_acl,black_acl,white_blunders,black_blunders,white_mistakes,black_mistakes,final_eval,max_white_advantage,max_black_advantage,game_sharpness
0,1,38,13.67,15.11,0,0,0,0,54,73,-26,26.43
1,2,13,12.00,12.33,0,0,0,0,55,55,14,11.54
2,3,106,312.33,203.29,7,2,0,0,-11544,93,-11544,2286.13


## 7. Merge All Sources into One DataFrame
Join on `event_id` (the shared game identifier across all three files).

In [8]:
df = merge_datasets(df_pgn, df_uci, df_sf)
df.head(3)

Merged DataFrame shape: (23636, 27)
Columns: ['event_id', 'white_elo', 'black_elo', 'result', 'num_moves', 'white_castled', 'black_castled', 'white_castle_side', 'black_castle_side', 'num_captures', 'termination', 'moves_san', 'eco_code', 'opening_name', 'eco_family', 'moves_uci', 'total_half_moves', 'white_acl', 'black_acl', 'white_blunders', 'black_blunders', 'white_mistakes', 'black_mistakes', 'final_eval', 'max_white_advantage', 'max_black_advantage', 'game_sharpness']


,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,...,white_acl,black_acl,white_blunders,black_blunders,white_mistakes,black_mistakes,final_eval,max_white_advantage,max_black_advantage,game_sharpness
0,1,2354,2411,1/2-1/2,19,True,True,kingside,kingside,9,...,13.67,15.11,0,0,0,0,54,73,-26,26.43
1,2,2523,2460,1/2-1/2,6,True,False,kingside,none,2,...,12.00,12.33,0,0,0,0,55,55,14,11.54
2,3,1915,1999,0-1,53,True,True,kingside,kingside,23,...,312.33,203.29,7,2,0,0,-11544,93,-11544,2286.13


## 8. Feature Engineering
Derive additional features that carry useful signal for both tasks:

| Feature | Formula | Rationale |
|---|---|---|
| `elo_gap` | `white_elo - black_elo` | Relative strength difference |
| `avg_elo` | `(white_elo + black_elo) / 2` | Overall game quality proxy |
| `elo_bucket_white` | Binned white Elo | Target variable for Elo classification task |
| `elo_bucket_black` | Binned black Elo | Target variable for Elo classification task |
| `acl_gap` | `white_acl - black_acl` | Relative accuracy difference |
| `winner` | Encoded result | Target variable for winner prediction task |

In [9]:
engineered_df = engineer_features(df)
engineered_df.head(3)

Engineered features added.

Elo bucket distribution (White):
elo_bucket_white
Beginner            0
Intermediate      169
Advanced         3829
Expert          15216
Master           4422


,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,...,max_white_advantage,max_black_advantage,game_sharpness,elo_bucket_white,elo_bucket_black,elo_bucket_white_categorical,elo_bucket_black_categorical,acl_gap,winner_binary,winner_multiclass
0,1,2354,2411,1/2-1/2,19,True,True,kingside,kingside,9,...,73,-26,26.43,Expert,Expert,3,3,-1.44,NaN,1
1,2,2523,2460,1/2-1/2,6,True,False,kingside,none,2,...,55,14,11.54,Master,Expert,4,3,-0.33,NaN,1
2,3,1915,1999,0-1,53,True,True,kingside,kingside,23,...,93,-11544,2286.13,Advanced,Advanced,2,2,109.04,0.0,0


## 10. Integrate Lichess Dataset
Loads `chess_games.csv` (Lichess casual/club games), harmonises its columns to match
the PGN pipeline schema, then concatenates both datasets into one unified DataFrame.

**Column mapping:**

| Unified column | PGN source | Lichess source |
|---|---|---|
| `white_elo` | `white_elo` | `white_rating` |
| `black_elo` | `black_elo` | `black_rating` |
| `num_moves` | `num_moves` | `turns // 2` |
| `termination` | `termination` | `victory_status` |
| `eco_code` | `eco_code` | `opening_code` |
| `opening_name` | `opening_name` | `opening_fullname` |
| `eco_family` | `eco_family` | `opening_code[0]` |
| `winner_multiclass` | encoded from `result` | encoded from `winner` |
| `has_stockfish` | `True` | `False` |

Stockfish columns (`white_acl`, `black_acl`, blunders, etc.) will be `NaN`
for all Lichess rows — handled during preprocessing via imputation.

In [10]:
load_lichess(LICHESS_PATH, LICHESS_SF_PATH)
df_combined = integrate_datasets(df, LICHESS_PATH, LICHESS_SF_PATH)
df_combined.tail(3)

Loaded 20,058 games from '../data/raw/chess_games.csv'
Loading Stockfish features from '../data/raw/lichess_stockfish.csv'...
Extracted Stockfish features for 20,040 games
       white_acl  black_acl  white_blunders  black_blunders
count   20040.00   20040.00        20040.00        20040.00
mean      150.13     102.00            3.13            3.00
std       196.93     111.80            2.53            2.52
min         0.00       0.00            0.00            0.00
25%        48.67      43.77            1.00            1.00
50%        86.96      73.00            3.00            3.00
75%       182.54     128.28            5.00            4.00
max      4000.00    2136.00           20.00           20.00
Extracted Stockfish features for 20,040 games

Lichess harmonised shape: (20058, 35)
has_stockfish = True : 20,040
has_stockfish = False: 18
Loaded 20,058 games from '../data/raw/chess_games.csv'
Loading Stockfish features from '../data/raw/lichess_stockfish.csv'...
Extracted Stockfish f

,event_id,white_elo,black_elo,result,num_moves,white_castled,black_castled,white_castle_side,black_castle_side,num_captures,...,game_sharpness,source,has_stockfish,elo_bucket_white,elo_bucket_black,elo_bucket_white_categorical,elo_bucket_black_categorical,winner_multiclass,winner_binary,acl_gap
43691,43692,1219,1286,NaN,17,NaN,NaN,NaN,NaN,NaN,...,681.81,lichess_csv,True,Intermediate,Intermediate,1,1,2.0,1.0,-115.65
43692,43693,1360,1227,NaN,54,NaN,NaN,NaN,NaN,NaN,...,594.35,lichess_csv,True,Intermediate,Intermediate,1,1,2.0,1.0,12.68
43693,43694,1235,1339,NaN,39,NaN,NaN,NaN,NaN,NaN,...,638.91,lichess_csv,True,Intermediate,Intermediate,1,1,0.0,0.0,263.31


## 11.1. Data Validation Report (Games with stockfish)

In [11]:
validation_report(df)

DATA VALIDATION REPORT

Shape: 23,636 rows × 27 columns

── Data Types ──────────────────────────────────────────────
event_id                 int64
white_elo                int64
black_elo                int64
result                  object
num_moves                int64
white_castled             bool
black_castled             bool
white_castle_side       object
black_castle_side       object
num_captures             int64
termination             object
moves_san               object
eco_code                object
opening_name            object
eco_family              object
moves_uci               object
total_half_moves         int64
white_acl              float64
black_acl              float64
white_blunders           int64
black_blunders           int64
white_mistakes           int64
black_mistakes           int64
final_eval               int64
max_white_advantage      int64
max_black_advantage      int64
game_sharpness         float64

── Missing Values ──────────────────────────

## 11.2. Data Validation Report (Games with no stockfish)

In [12]:
validation_report(load_lichess(LICHESS_PATH, LICHESS_SF_PATH))

Loaded 20,058 games from '../data/raw/chess_games.csv'
Loading Stockfish features from '../data/raw/lichess_stockfish.csv'...
Extracted Stockfish features for 20,040 games
       white_acl  black_acl  white_blunders  black_blunders
count   20040.00   20040.00        20040.00        20040.00
mean      150.13     102.00            3.13            3.00
std       196.93     111.80            2.53            2.52
min         0.00       0.00            0.00            0.00
25%        48.67      43.77            1.00            1.00
50%        86.96      73.00            3.00            3.00
75%       182.54     128.28            5.00            4.00
max      4000.00    2136.00           20.00           20.00
Extracted Stockfish features for 20,040 games

Lichess harmonised shape: (20058, 35)
has_stockfish = True : 20,040
has_stockfish = False: 18
DATA VALIDATION REPORT

Shape: 20,058 rows × 35 columns

── Data Types ──────────────────────────────────────────────
event_id                     

## 11.3. Data Validation Report (All games)

In [13]:
validation_report(df_combined)

DATA VALIDATION REPORT

Shape: 43,694 rows × 36 columns

── Data Types ──────────────────────────────────────────────
event_id                           int64
white_elo                          int64
black_elo                          int64
result                            object
num_moves                          int64
white_castled                     object
black_castled                     object
white_castle_side                 object
black_castle_side                 object
num_captures                     float64
termination                       object
moves_san                         object
eco_code                          object
opening_name                      object
eco_family                        object
moves_uci                         object
total_half_moves                   int64
white_acl                        float64
black_acl                        float64
white_blunders                   float64
black_blunders                   float64
white_mistakes       

## 12. Save Final Dataset
Save the merged and engineered DataFrame to `merged_games.csv` — this is the input for EDA and modeling.
Save the DataFrame with stockfish only to `games.csv`

In [14]:
save_dataset(df_combined, OUTPUT_MERGED_PATH)
save_dataset(df, OUTPUT_PATH)

Saved 43,694 rows × 35 columns to '../data/processed/merged_games.csv'

Final columns:
['event_id', 'white_elo', 'black_elo', 'result', 'num_moves', 'white_castled', 'black_castled', 'white_castle_side', 'black_castle_side', 'num_captures', 'termination', 'moves_san', 'eco_code', 'opening_name', 'eco_family', 'total_half_moves', 'white_acl', 'black_acl', 'white_blunders', 'black_blunders', 'white_mistakes', 'black_mistakes', 'final_eval', 'max_white_advantage', 'max_black_advantage', 'game_sharpness', 'source', 'has_stockfish', 'elo_bucket_white', 'elo_bucket_black', 'elo_bucket_white_categorical', 'elo_bucket_black_categorical', 'winner_multiclass', 'winner_binary', 'acl_gap']
Saved 23,636 rows × 26 columns to '../data/processed/games.csv'

Final columns:
['event_id', 'white_elo', 'black_elo', 'result', 'num_moves', 'white_castled', 'black_castled', 'white_castle_side', 'black_castle_side', 'num_captures', 'termination', 'moves_san', 'eco_code', 'opening_name', 'eco_family', 'total_ha